# 03. Tuning & Run (Optuna + N_REPEATS 최종 학습)

01에서 만든 FuxiCTR parquet을 사용해 `MODEL`을 튜닝하고, best params로 N_REPEATS번 재학습 뒤
메트릭과 test 예측을 저장한다.

흐름:
1. `DATASET` / `MODEL` / `N_TRIALS` / `N_REPEATS` 스위치만 설정
2. `configs/models/{MODEL}_base.yaml` + `configs/tuning/{MODEL}_search.yaml` 로드
3. Optuna study (SQLite `artifacts/tuning/...` 저장 → 중단/재개 가능)
4. best params로 N_REPEATS번 재학습 → 메트릭 + test 예측 저장
5. 저장된 run 요약 출력

산출물:
- `artifacts/runs/{dataset_id}/{model_id}.model` (FuxiCTR best checkpoint)
- `artifacts/runs/{dataset_id}/{model_id}.metrics.json` / `.params.json`
- `artifacts/predictions/{dataset_id}/{MODEL}/{run_id}/preds.parquet`
- `artifacts/tuning/{dataset_id}/{MODEL}/study.db` (Optuna, 재개 가능)

In [1]:
import sys, json, copy
from pathlib import Path
import pandas as pd

_HERE = Path.cwd()
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))
from utils import nb_utils as U
U.add_fuxictr_to_path()

import optuna
print('optuna =', optuna.__version__)

optuna = 4.8.0


## 1. 스위치

다른 실험을 돌리려면 이 셀만 바꾸면 된다.

In [2]:
DATASET_ID = 'AmazonElectronics_x1'   # 01에서 빌드된 dataset_id
MODEL      = 'DIN'                     # configs/models/{MODEL}_base.yaml 에 정의된 이름
N_TRIALS   = 3                        # Optuna trial 수 (smoke test는 3~5)
N_REPEATS  = 2                         # best params로 재학습할 횟수 (seed만 다름)
SEEDS      = [2026, 2027, 2028, 2029, 2030][:N_REPEATS]
TUNE_EPOCHS = 3                        # 튜닝 trial당 epoch (빨리 탐색)

# study DB (SQLite) 위치 — 존재하면 자동 resume
study_dir = U.ARTIFACT_ROOT / 'tuning' / DATASET_ID / MODEL
study_dir.mkdir(parents=True, exist_ok=True)
STUDY_DB = study_dir / 'study.db'
STUDY_NAME = f'{MODEL}__{DATASET_ID}'
print(f'dataset={DATASET_ID}  model={MODEL}  trials={N_TRIALS}  repeats={N_REPEATS}')
print(f'study  = sqlite:///{STUDY_DB}')

dataset=AmazonElectronics_x1  model=DIN  trials=3  repeats=2
study  = sqlite:////NAS/hyunwoong/Hayanmind-project/notebook/artifacts/tuning/AmazonElectronics_x1/DIN/study.db


## 2. Base config + search space 로드

- `load_base_config(MODEL)` → `DIN_base.yaml`의 Base + MODEL 섹션 + 기본값 머지
- `load_search_space(MODEL)` → `DIN_search.yaml` 의 탐색 공간

In [3]:
base_cfg = U.load_base_config(MODEL)
search_space = U.load_search_space(MODEL)

# feature_map이 있어야 data 경로 검증이 가능
fmap = U.load_feature_map(DATASET_ID)
print(f'base_cfg keys (일부): {sorted(base_cfg)[:12]} ...')
print(f'search_space       : {list(search_space)}')
print(f'feature_map labels : {fmap["labels"]}  total_features={fmap["total_features"]}')

base_cfg keys (일부): ['attention_dropout', 'attention_hidden_activations', 'attention_hidden_units', 'attention_output_activation', 'batch_norm', 'batch_size', 'debug_mode', 'din_sequence_field', 'din_target_field', 'din_use_softmax', 'dnn_activations', 'dnn_hidden_units'] ...
search_space       : ['learning_rate', 'embedding_dim', 'dnn_hidden_units', 'net_dropout', 'attention_dropout', 'batch_size']
feature_map labels : ['label']  total_features=236971


## 3. Optuna objective

- 각 trial마다 `{MODEL}_tune_t{trial_number}` 로 `run_training` 호출 (test 예측 저장 X)
- valid AUC를 반환 → Optuna가 maximize
- 실패한 trial은 `optuna.TrialPruned` 로 스킵

⚠️ 튜닝 중에도 checkpoint는 `artifacts/runs/{dataset_id}/` 아래에 쌓이므로,
나중에 정리하려면 `{MODEL}_tune_*` 패턴으로 삭제하면 된다.

In [4]:
def objective(trial: 'optuna.Trial') -> float:
    suggested = U.suggest_from_space(trial, search_space)
    params = copy.deepcopy(base_cfg)
    params.update(suggested)
    params['epochs'] = TUNE_EPOCHS
    params['save_best_only'] = False  # 튜닝 중 checkpoint IO 최소화
    run_id = f'tune_t{trial.number:03d}_s{params.get("seed", 2026)}'
    try:
        summary = U.run_training(
            dataset_id=DATASET_ID,
            model_name=MODEL,
            params=params,
            run_id=run_id,
            save_test_preds=False,
        )
    except Exception as e:
        print(f'[trial {trial.number}] failed: {e!r}')
        raise optuna.TrialPruned() from e
    auc = summary.get('valid_AUC')
    if auc is None:
        raise optuna.TrialPruned()
    print(f'[trial {trial.number}] valid_AUC={auc:.5f}  params={suggested}')
    return float(auc)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=f'sqlite:///{STUDY_DB}',
    direction='maximize',
    load_if_exists=True,
)
print(f'study loaded. prior trials = {len(study.trials)}')

[I 2026-04-16 12:11:07,833] Using an existing study with name 'DIN__AmazonElectronics_x1' instead of creating a new one.


study loaded. prior trials = 2


In [5]:
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
print('\n=== Optuna summary ===')
print('best value  =', study.best_value)
print('best params =', study.best_params)
df_trials = study.trials_dataframe(attrs=('number','value','state','params','datetime_complete'))
df_trials.tail(min(20, len(df_trials)))

2026-04-16 12:11:09,505 P2243158 INFO FuxiCTR version: 2.3.9
2026-04-16 12:11:09,507 P2243158 INFO Run tune_t002_s2026: {
    "attention_dropout": "0.06185307020001543",
    "attention_hidden_activations": "Dice",
    "attention_hidden_units": "[64]",
    "attention_output_activation": "None",
    "batch_norm": "False",
    "batch_size": "8192",
    "data_format": "parquet",
    "data_root": "/NAS/hyunwoong/Hayanmind-project/data",
    "dataset_id": "AmazonElectronics_x1",
    "debug_mode": "False",
    "din_sequence_field": "[('item_history', 'cate_history')]",
    "din_target_field": "[('item_id', 'cate_id')]",
    "din_use_softmax": "False",
    "dnn_activations": "relu",
    "dnn_hidden_units": "[256, 128]",
    "early_stop_patience": "2",
    "embedding_dim": "64",
    "embedding_regularizer": "0",
    "epochs": "3",
    "eval_steps": "None",
    "feature_config": "None",
    "feature_specs": "[{'name': 'item_history', 'feature_encoder': None}, {'name': 'cate_history', 'feature_en

 99%|█████████▉| 285/287 [00:16<00:00, 13.24it/s]

2026-04-16 12:11:36,845 P2243158 INFO Train loss: 0.536376
2026-04-16 12:11:36,847 P2243158 INFO Evaluation @epoch 1 - batch 287: 


100%|██████████| 32/32 [00:01<00:00, 17.27it/s]


2026-04-16 12:11:38,914 P2243158 INFO [Metrics] AUC: 0.844292


...

2026-04-16 12:15:19,434 P2243158 INFO [Metrics] AUC: 0.854014 - logloss: 0.478269
2026-04-16 12:15:19,437 P2243158 INFO *** Test evaluation (run=tune_t004_s2026) ***
2026-04-16 12:15:19,438 P2243158 INFO Loading datasets...
2026-04-16 12:15:21,050 P2243158 INFO Test samples: total/384806, blocks/1
2026-04-16 12:15:21,052 P2243158 INFO Loading test data done.


100%|██████████| 188/188 [00:03<00:00, 57.47it/s]


2026-04-16 12:15:24,612 P2243158 INFO [Metrics] AUC: 0.835699 - logloss: 0.551172
[I 2026-04-16 12:15:24,672] Trial 4 finished with value: 0.8540137399733095 and parameters: {'learning_rate': 0.0004029788665423237, 'embedding_dim': 32, 'dnn_hidden_units': '[1024, 512, 256]', 'net_dropout': 0.2779865882586993, 'attention_dropout': 0.2531238078843239, 'batch_size': 2048}. Best is trial 2 with value: 0.8635455267152676.


[run tune_t004_s2026] valid=OrderedDict([('AUC', 0.8540137399733095), ('logloss', 0.4782685044492419)])  test=OrderedDict([('AUC', 0.8356987260674158), ('logloss', 0.5511715945502569)])
[run tune_t004_s2026] artifacts -> /NAS/hyunwoong/Hayanmind-project/notebook/artifacts/runs/AmazonElectronics_x1/DIN_tune_t004_s2026.*
[trial 4] valid_AUC=0.85401  params={'learning_rate': 0.0004029788665423237, 'embedding_dim': 32, 'dnn_hidden_units': [1024, 512, 256], 'net_dropout': 0.2779865882586993, 'attention_dropout': 0.2531238078843239, 'batch_size': 2048}

=== Optuna summary ===
best value  = 0.8635455267152676
best params = {'learning_rate': 0.002474107864693061, 'embedding_dim': 64, 'dnn_hidden_units': '[256, 128]', 'net_dropout': 0.3944041720918892, 'attention_dropout': 0.06185307020001543, 'batch_size': 8192}


,number,value,state,params_attention_dropout,params_batch_size,params_dnn_hidden_units,params_embedding_dim,params_learning_rate,params_net_dropout,datetime_complete
0,0,0.852111,COMPLETE,0.252576,4096,"[512, 256, 128]",16,0.000255,0.089314,2026-04-16 12:06:57.637472
1,1,0.855176,COMPLETE,0.283607,4096,"[256, 128]",16,0.000506,0.426405,2026-04-16 12:08:02.317216
2,2,0.863546,COMPLETE,0.061853,8192,"[256, 128]",64,0.002474,0.394404,2026-04-16 12:12:21.687139
3,3,0.857037,COMPLETE,0.053560,2048,"[256, 128]",32,0.002285,0.085129,2026-04-16 12:13:54.526782
4,4,0.854014,COMPLETE,0.253124,2048,"[1024, 512, 256]",32,0.000403,0.277987,2026-04-16 12:15:24.655252


## 4. best params로 최종 학습 (N_REPEATS)

seed만 다르게 `N_REPEATS`번 재학습한다. 이때는 `save_test_preds=True` 로
test 예측을 parquet에 저장 → 04 분석 노트북에서 바로 로드 가능.

In [6]:
final_summaries = []
for seed in SEEDS:
    params = copy.deepcopy(base_cfg)
    params.update(U.decode_best_params(study.best_params))
    params['seed'] = seed
    run_id = U.make_run_id(MODEL, DATASET_ID, seed)
    print(f'\n=== FINAL RUN seed={seed}  run_id={run_id} ===')
    summary = U.run_training(
        dataset_id=DATASET_ID,
        model_name=MODEL,
        params=params,
        run_id=run_id,
        save_test_preds=True,
    )
    final_summaries.append(summary)

final_df = pd.DataFrame(final_summaries)
print('\n=== final runs ===')
final_df

2026-04-16 12:15:24,746 P2243158 INFO FuxiCTR version: 2.3.9
2026-04-16 12:15:24,748 P2243158 INFO Run 20260416-121524_DIN_AmazonElectronics_x1_s2026: {
    "attention_dropout": "0.06185307020001543",
    "attention_hidden_activations": "Dice",
    "attention_hidden_units": "[64]",
    "attention_output_activation": "None",
    "batch_norm": "False",
    "batch_size": "8192",
    "data_format": "parquet",
    "data_root": "/NAS/hyunwoong/Hayanmind-project/data",
    "dataset_id": "AmazonElectronics_x1",
    "debug_mode": "False",
    "din_sequence_field": "[('item_history', 'cate_history')]",
    "din_target_field": "[('item_id', 'cate_id')]",
    "din_use_softmax": "False",
    "dnn_activations": "relu",
    "dnn_hidden_units": "[256, 128]",
    "early_stop_patience": "2",
    "embedding_dim": "64",
    "embedding_regularizer": "0",
    "epochs": "20",
    "eval_steps": "None",
    "feature_config": "None",
    "feature_specs": "[{'name': 'item_history', 'feature_encoder': None}, {'na


=== FINAL RUN seed=2026  run_id=20260416-121524_DIN_AmazonElectronics_x1_s2026 ===


2026-04-16 12:15:32,006 P2243158 INFO Train samples: total/2347550, blocks/1
2026-04-16 12:15:33,127 P2243158 INFO Validation samples: total/261214, blocks/1
2026-04-16 12:15:33,129 P2243158 INFO Loading train and validation data done.
2026-04-16 12:15:33,350 P2243158 INFO Total number of parameters: 15314306.
2026-04-16 12:15:33,352 P2243158 INFO Start training: 287 batches/epoch
2026-04-16 12:15:33,353 P2243158 INFO ************ Epoch=1 start ************


100%|█████████▉| 286/287 [00:14<00:00, 15.03it/s]

2026-04-16 12:15:48,049 P2243158 INFO Train loss: 0.536376
2026-04-16 12:15:48,051 P2243158 INFO Evaluation @epoch 1 - batch 287: 


...

2026-04-16 12:18:46,812 P2243158 INFO [Metrics] AUC: 0.862924 - logloss: 0.463801
2026-04-16 12:18:46,817 P2243158 INFO *** Test evaluation (run=20260416-121659_DIN_AmazonElectronics_x1_s2027) ***
2026-04-16 12:18:46,819 P2243158 INFO Loading datasets...
2026-04-16 12:18:48,567 P2243158 INFO Test samples: total/384806, blocks/1
2026-04-16 12:18:48,569 P2243158 INFO Loading test data done.


100%|██████████| 47/47 [00:03<00:00, 13.16it/s]


2026-04-16 12:18:52,481 P2243158 INFO [Metrics] AUC: 0.845535 - logloss: 0.520752
2026-04-16 12:18:52,486 P2243158 INFO Loading datasets...
2026-04-16 12:18:54,166 P2243158 INFO Test samples: total/384806, blocks/1
2026-04-16 12:18:54,168 P2243158 INFO Loading test data done.


100%|██████████| 47/47 [00:03<00:00, 13.07it/s]
[preds] saved 384,806 rows -> /NAS/hyunwoong/Hayanmind-project/notebook/artifacts/predictions/AmazonElectronics_x1/DIN/20260416-121659_DIN_AmazonElectronics_x1_s2027/preds.parquet
[run 20260416-121659_DIN_AmazonElectronics_x1_s2027] valid=OrderedDict([('AUC', 0.8629237503448335), ('logloss', 0.46380101073939767)])  test=OrderedDict([('AUC', 0.8455350325829702), ('logloss', 0.5207521291036592)])
[run 20260416-121659_DIN_AmazonElectronics_x1_s2027] artifacts -> /NAS/hyunwoong/Hayanmind-project/notebook/artifacts/runs/AmazonElectronics_x1/DIN_20260416-121659_DIN_AmazonElectronics_x1_s2027.*
[run 20260416-121659_DIN_AmazonElectronics_x1_s2027] preds    -> /NAS/hyunwoong/Hayanmind-project/notebook/artifacts/predictions/AmazonElectronics_x1/DIN/20260416-121659_DIN_AmazonElectronics_x1_s2027/preds.parquet

=== final runs ===


,model,dataset_id,run_id,model_id,train_seconds,valid_AUC,valid_logloss,test_AUC,test_logloss
0,DIN,AmazonElectronics_x1,20260416-121524_DIN_AmazonElectronics_x1_s2026,DIN_20260416-121524_DIN_AmazonElectronics_x1_s...,73.11,0.864903,0.460568,0.852022,0.482683
1,DIN,AmazonElectronics_x1,20260416-121659_DIN_AmazonElectronics_x1_s2027,DIN_20260416-121659_DIN_AmazonElectronics_x1_s...,97.06,0.862924,0.463801,0.845535,0.520752


## 5. 저장된 run 요약 출력 (list_runs)

`artifacts/runs/{DATASET_ID}/` 아래에 쌓인 모든 run(튜닝 trial 포함)을 메트릭 표로.

In [7]:
runs = U.list_runs(DATASET_ID)
cols = [c for c in ['dataset','model','run_id','metric.valid_AUC','metric.test_AUC','metric.valid_logloss','metric.test_logloss','metric.train_seconds'] if c in runs.columns]
runs[cols].sort_values(cols[-2] if 'metric.test_AUC' in cols else 'run_id', ascending=False).head(30)

,dataset,model,run_id,metric.valid_AUC,metric.test_AUC,metric.valid_logloss,metric.test_logloss,metric.train_seconds
5,AmazonElectronics_x1,DIN,tune_t003_s2026,0.857037,0.834420,0.474442,0.578799,76.36
6,AmazonElectronics_x1,DIN,tune_t004_s2026,0.854014,0.835699,0.478269,0.551172,73.49
2,AmazonElectronics_x1,DIN,tune_t000_s2026,0.852111,0.836035,0.478116,0.521070,49.87
1,AmazonElectronics_x1,DIN,20260416-121659_DIN_AmazonElectronics_x1_s2027,0.862924,0.845535,0.463801,0.520752,97.06
4,AmazonElectronics_x1,DIN,tune_t002_s2026,0.863546,0.849563,0.464459,0.512983,55.22
3,AmazonElectronics_x1,DIN,tune_t001_s2026,0.855176,0.839740,0.473655,0.511686,50.13
0,AmazonElectronics_x1,DIN,20260416-121524_DIN_AmazonElectronics_x1_s2026,0.864903,0.852022,0.460568,0.482683,73.11
